In [ ]:
"""
exponents_analysis.py
---------------------
Fast, accurate extraction of beta, gamma and tau using only
the methods that are trustworthy at finite L.

Methods used
------------
beta  — quotient method: P∞(2L)/P∞(L) ~ 2^(-beta/nu) at pc
gamma — quotient method: chi(2L)/chi(L) ~ 2^(gamma/nu) at pc
tau   — truncated MLE (Clauset-Shalizi-Newman 2009 + finite-size
        correction): accounts for the hard cutoff at s ~ L² so
        the estimator is not biased by the missing tail

Fixes vs previous version
--------------------------
1. P_WINDOW centred exactly on pc — quotient method requires p=pc
2. Truncated power law MLE for tau — corrects for finite-size cutoff
   at s_max ~ L², gives less biased tau than the naive MLE
3. smin floor of 5 for MLE — avoids bias from discrete corrections
   at very small s
4. Tighter plateau criterion (spread < 0.02) for smin stability scan
5. Bootstrap uncertainties on quotient estimates via per-run resampling
   (std across pairs was not a proper statistical uncertainty)
6. Correct sign in beta quotient formula (was missing minus sign)
7. Gamma uses quotient only — no mixed-side log-log fit

What is NOT used
----------------
Direct log-log fits for beta/gamma — window-sensitive, unreliable
Log-log fit for tau — needs many decades, corrupted at finite L

Usage
-----
    python exponents_analysis.py

Edit the configuration block below before running.
PC_OVERRIDE and NU_OVERRIDE should be your FSS values.
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from joblib import Parallel, delayed

from observables import generate_lattice, spanning_info
from algorithms  import hoshen_kopelman
from analysis    import THEORY

try:
    from tqdm import tqdm as _tqdm
    _TQDM = True
except ImportError:
    _TQDM = False

# =============================================================================
# Configuration
# =============================================================================

PC_OVERRIDE = 0.59296       # your FSS pc from the overnight run
NU_OVERRIDE = 1.378           # exact theoretical value gives best accuracy
                            # set to 1.378 to use your measured value instead

# Lattice sizes — must include doubling pairs e.g. (64,128), (128,256), (256,512)
L_FSS = [16, 32, 64, 128, 256, 512]

# Runs per p value for quotient method
RUNS_QUOTIENT = 1000

# p window — centred exactly on pc, narrow range
# Only 7 points needed; the one closest to pc is used for the quotient
N_P_WINDOW  = 7
P_HALFWIDTH = 0.003         # window extends pc ± 0.003

# MLE tau settings
L_MLE      = 512            # largest available L gives best tau
RUNS_MLE   = 2000           # more runs = more clusters = better MLE
SMIN_FLOOR = 5              # minimum smin to avoid discrete bias at small s

# Bootstrap resamples for quotient uncertainty
N_BOOT = 300

SPAN_MODE = "LR"
BASE_SEED = 42
FIG_DIR   = "figures"

# =============================================================================
# Helpers
# =============================================================================

def _save(fig, name):
    os.makedirs(FIG_DIR, exist_ok=True)
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved: {path}")


def _style(ax, xlabel, ylabel, title):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)


def _single_run_full(L, p, span_mode, seed):
    """
    One HK realisation returning Pinf, chi, and all raw cluster sizes.

    span_mode is respected consistently with simulation.py:
      "LR"  — only clusters spanning left-to-right count as spanning
      "TB"  — only clusters spanning top-to-bottom count as spanning
      "ANY" — clusters spanning in either direction count as spanning

    spanning_info() always returns roots for LR | TB (i.e. ANY), so
    we must gate it on span_mode explicitly, otherwise Pinf is
    overestimated and chi is underestimated relative to the main run.
    """
    rng     = np.random.default_rng(seed)
    lattice = generate_lattice(L, p, rng=rng)
    labels, cluster_sizes = hoshen_kopelman(lattice)
    spans_lr, spans_tb, all_spanning_roots = spanning_info(labels)

    # Gate spanning roots on span_mode — must match simulation.py exactly
    if span_mode == "LR":
        active_roots = all_spanning_roots if spans_lr else set()
    elif span_mode == "TB":
        active_roots = all_spanning_roots if spans_tb else set()
    elif span_mode == "ANY":
        active_roots = all_spanning_roots   # already LR | TB
    else:
        raise ValueError(f"Unknown span_mode: {span_mode!r}")

    N    = L * L
    pinf = sum(cluster_sizes.get(r, 0) for r in active_roots) / N

    finite = np.array([s for r, s in cluster_sizes.items()
                       if r not in active_roots], dtype=np.float64)
    chi = float((finite**2).sum() / finite.sum()) if finite.size > 0 else 0.0

    # Raw non-spanning cluster sizes >= 2 (used for MLE tau)
    raw_sizes = [s for r, s in cluster_sizes.items()
                 if r not in active_roots and s >= 2]

    return {"pinf": pinf, "chi": chi, "sizes": raw_sizes}


def _simulate_at_p(L, p, runs, seed_offset=0, desc=None):
    """
    Run `runs` realisations at a single (L, p).
    Returns mean Pinf, mean chi, all raw cluster sizes, and per-run arrays.

    Progress bar tracks actual job completions, not job submissions.
    """
    seeds = BASE_SEED + seed_offset + np.arange(runs, dtype=int)
    jobs  = (delayed(_single_run_full)(L, p, SPAN_MODE, int(s)) for s in seeds)

    if _TQDM and desc:
        raw     = Parallel(n_jobs=-1, prefer="threads", return_as="generator")(jobs)
        results = list(_tqdm(
            raw, total=runs, desc=desc,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
        ))
    else:
        results = Parallel(n_jobs=-1, prefer="threads")(jobs)

    pinf_arr  = np.array([r["pinf"] for r in results])
    chi_arr   = np.array([r["chi"]  for r in results])
    all_sizes = []
    for r in results:
        all_sizes.extend(r["sizes"])

    return {
        "Pinf":      float(pinf_arr.mean()),
        "chi":       float(chi_arr.mean()),
        "sizes":     np.array(all_sizes, dtype=float),
        "Pinf_runs": pinf_arr,
        "chi_runs":  chi_arr,
    }


# =============================================================================
# Quotient method — beta and gamma with bootstrap uncertainties
# =============================================================================

def _quotient_from_runs(Pinf_runs1, Pinf_runs2, chi_runs1, chi_runs2,
                        L1, L2, nu, rng=None):
    """
    Compute beta and gamma from one (optionally bootstrapped) sample.

    At criticality:
        P∞(pc, L)  ~ L^(-beta/nu)  =>  beta  = -nu * log(P∞(2L)/P∞(L)) / log(2)
        chi(pc, L) ~ L^(gamma/nu)  =>  gamma =  nu * log(chi(2L)/chi(L)) / log(2)

    The minus sign in beta is essential: P∞ decreases with L at pc,
    so the ratio is < 1 and its log is negative.
    """
    if rng is not None:
        idx1 = rng.integers(0, len(Pinf_runs1), len(Pinf_runs1))
        idx2 = rng.integers(0, len(Pinf_runs2), len(Pinf_runs2))
        Pinf1 = Pinf_runs1[idx1].mean()
        Pinf2 = Pinf_runs2[idx2].mean()
        chi1  = chi_runs1[idx1].mean()
        chi2  = chi_runs2[idx2].mean()
    else:
        Pinf1 = Pinf_runs1.mean()
        Pinf2 = Pinf_runs2.mean()
        chi1  = chi_runs1.mean()
        chi2  = chi_runs2.mean()

    beta = gamma = np.nan
    log_ratio = np.log(L2 / L1)   # = log(2) for exact doubling pairs

    if Pinf1 > 0 and Pinf2 > 0:
        ratio = Pinf2 / Pinf1
        if ratio > 0:
            beta = -np.log(ratio) / log_ratio * nu

    if chi1 > 0 and chi2 > 0:
        ratio = chi2 / chi1
        if ratio > 0:
            gamma = np.log(ratio) / log_ratio * nu

    return beta, gamma


def run_quotient(pc, nu):
    """
    Simulate at p values centred on pc for each L in L_FSS.
    Extract beta and gamma from ratios at paired L values.
    Bootstrap uncertainties from per-run resampling.
    """
    p_window = np.linspace(pc - P_HALFWIDTH, pc + P_HALFWIDTH, N_P_WINDOW)
    k_pc     = int(np.argmin(np.abs(p_window - pc)))
    p_use    = float(p_window[k_pc])

    print(f"\n  P window: [{p_window[0]:.5f}, {p_window[-1]:.5f}]  "
          f"{N_P_WINDOW} points")
    print(f"  Using p = {p_use:.5f}  (offset from pc = {p_use - pc:+.6f})")

    # Simulate each L at p_use
    data = {}
    for i, L_val in enumerate(L_FSS):
        res = _simulate_at_p(
            L_val, p_use, RUNS_QUOTIENT,
            seed_offset=30000 + i * 10000,
            desc=f"  Quotient L={L_val:4d}",
        )
        data[L_val] = res
        print(f"    L={L_val:4d}  Pinf={res['Pinf']:.5f}  chi={res['chi']:.2f}")

    # Identify exact doubling pairs
    pairs = [(L_FSS[i], L_FSS[i+1]) for i in range(len(L_FSS)-1)
             if L_FSS[i+1] == 2 * L_FSS[i]]

    beta_pairs, gamma_pairs = [], []
    beta_errs,  gamma_errs  = [], []
    pair_labels = []
    rng = np.random.default_rng(BASE_SEED + 60000)

    for L1, L2 in pairs:
        d1, d2 = data[L1], data[L2]

        # Point estimate
        beta_pt, gamma_pt = _quotient_from_runs(
            d1["Pinf_runs"], d2["Pinf_runs"],
            d1["chi_runs"],  d2["chi_runs"],
            L1, L2, nu,
        )

        # Bootstrap uncertainty
        b_boot = np.empty(N_BOOT)
        g_boot = np.empty(N_BOOT)
        for b in range(N_BOOT):
            b_boot[b], g_boot[b] = _quotient_from_runs(
                d1["Pinf_runs"], d2["Pinf_runs"],
                d1["chi_runs"],  d2["chi_runs"],
                L1, L2, nu, rng=rng,
            )
        b_std = float(np.nanstd(b_boot))
        g_std = float(np.nanstd(g_boot))

        beta_pairs.append(beta_pt)
        gamma_pairs.append(gamma_pt)
        beta_errs.append(b_std)
        gamma_errs.append(g_std)
        pair_labels.append(f"L={L1}→{L2}")

        print(f"    beta  L={L1}->{L2}: {beta_pt:.4f} ± {b_std:.4f}  "
              f"(theory {THEORY['beta']:.4f})")
        print(f"    gamma L={L1}->{L2}: {gamma_pt:.4f} ± {g_std:.4f}  "
              f"(theory {THEORY['gamma']:.4f})")

    # Weighted mean across pairs (weight = 1/sigma^2)
    def _weighted_mean(vals, stds):
        vals = np.array(vals); stds = np.array(stds)
        valid = np.isfinite(vals) & np.isfinite(stds) & (stds > 0)
        if not valid.any():
            return np.nan, np.nan
        w   = 1.0 / stds[valid]**2
        mu  = (w * vals[valid]).sum() / w.sum()
        sig = 1.0 / np.sqrt(w.sum())
        return float(mu), float(sig)

    beta_mean,  beta_unc  = _weighted_mean(beta_pairs,  beta_errs)
    gamma_mean, gamma_unc = _weighted_mean(gamma_pairs, gamma_errs)

    return {
        "beta":        beta_mean,
        "gamma":       gamma_mean,
        "beta_std":    beta_unc,
        "gamma_std":   gamma_unc,
        "beta_pairs":  beta_pairs,
        "gamma_pairs": gamma_pairs,
        "beta_errs":   beta_errs,
        "gamma_errs":  gamma_errs,
        "pair_labels": pair_labels,
        "pairs":       pairs,
    }


# =============================================================================
# Truncated MLE for tau
# =============================================================================

def mle_tau_truncated(sizes, smin, smax):
    """
    MLE for a power law with hard upper cutoff at smax.

    The standard (untruncated) MLE assumes the distribution extends to
    infinity. At finite L the distribution is cut off at s ~ L², which
    biases the naive estimator downward — it sees the distribution drop
    off faster than a pure power law and compensates with a smaller tau.

    The truncated MLE solves the self-consistency equation:

        1/(tau-1) - smax^(1-tau)*ln(smax) / (1 - smax^(1-tau))
            = mean_i[ ln(s_i / (smin - 0.5)) ]

    via root finding (Brentq). This accounts for the missing tail and
    gives a less biased estimate.

    Parameters
    ----------
    sizes : array of raw cluster sizes
    smin  : minimum size to include (must be >= SMIN_FLOOR)
    smax  : hard upper cutoff ~ L^2

    Returns (tau, sigma, n)
    """
    s = sizes[(sizes >= smin) & (sizes <= smax)]
    n = len(s)
    if n < 10:
        return np.nan, np.nan, 0

    mean_log = float(np.mean(np.log(s / (smin - 0.5))))

    def equation(tau):
        if tau <= 1.0:
            return np.inf
        term1 = 1.0 / (tau - 1.0)
        r = float(smax) ** (1.0 - tau)
        # Correction term from truncation; r -> 0 for large tau so term2 -> 0
        term2 = (r * np.log(float(smax))) / (1.0 - r) if r < 1.0 else 0.0
        return term1 - term2 - mean_log

    try:
        tau = brentq(equation, 1.001, 10.0, xtol=1e-6, maxiter=200)
    except Exception:
        # Fallback to standard MLE if root-finding fails
        tau = 1.0 + n / np.sum(np.log(s / (smin - 0.5)))

    sigma = (tau - 1.0) / np.sqrt(n)
    return float(tau), float(sigma), int(n)


def run_mle_tau(pc):
    """
    Collect raw cluster sizes at pc for L_MLE.
    Apply truncated MLE with smin stability scan.
    Plateau criterion: spread < 0.02 over 3 consecutive smin values
    with at least 200 clusters — tighter than the previous 0.05.
    """
    p_window = np.linspace(pc - P_HALFWIDTH, pc + P_HALFWIDTH, N_P_WINDOW)
    p_use    = float(p_window[np.argmin(np.abs(p_window - pc))])
    smax     = float(L_MLE ** 2)

    print(f"\n  L={L_MLE}, runs={RUNS_MLE}, p={p_use:.5f}, "
          f"smax=L²={int(smax)}, smin_floor={SMIN_FLOOR}")

    res   = _simulate_at_p(L_MLE, p_use, RUNS_MLE,
                           seed_offset=90000,
                           desc=f"  MLE tau  L={L_MLE}")
    sizes = res["sizes"]
    print(f"  Total non-spanning clusters (s≥2): {len(sizes):,}")

    # Scan smin from SMIN_FLOOR to L^2/20
    smin_candidates = np.unique(np.round(
        np.logspace(
            np.log10(SMIN_FLOOR),
            np.log10(max(L_MLE**2 // 20, SMIN_FLOOR + 1)),
            40,
        )
    ).astype(int))

    tau_vals, sigma_vals, n_vals = [], [], []
    for smin in smin_candidates:
        tau, sigma, n = mle_tau_truncated(sizes, smin, smax)
        tau_vals.append(tau)
        sigma_vals.append(sigma)
        n_vals.append(n)

    tau_arr   = np.array(tau_vals)
    sigma_arr = np.array(sigma_vals)
    n_arr     = np.array(n_vals, dtype=float)

    # Find stable plateau: spread < 0.02 over 3 points, n >= 200
    best_tau, best_sigma, best_smin, best_idx = np.nan, np.nan, SMIN_FLOOR, 0
    for i in range(len(smin_candidates) - 2):
        window = tau_arr[i:i+3]
        if np.isfinite(window).all():
            if window.max() - window.min() < 0.02 and n_arr[i] >= 200:
                best_tau   = float(np.mean(window))
                best_sigma = float(sigma_arr[i])
                best_smin  = int(smin_candidates[i])
                best_idx   = i
                break

    if np.isnan(best_tau):
        best_tau, best_sigma, _ = mle_tau_truncated(sizes, SMIN_FLOOR, smax)
        best_smin = SMIN_FLOOR
        best_idx  = 0
        print(f"  Warning: no stable plateau found, using smin={SMIN_FLOOR}")

    best_n = int(n_arr[best_idx]) if best_idx < len(n_arr) else 0
    print(f"  tau = {best_tau:.4f} ± {best_sigma:.4f}  "
          f"smin={best_smin}  n={best_n:,} clusters  "
          f"(theory {THEORY['tau']:.4f})")

    return {
        "tau":        best_tau,
        "sigma":      best_sigma,
        "best_smin":  best_smin,
        "best_n":     best_n,
        "smin_vals":  smin_candidates,
        "tau_vals":   tau_arr,
        "sigma_vals": sigma_arr,
        "n_vals":     n_arr,
        "sizes":      sizes,
        "smax":       smax,
    }


# =============================================================================
# Plotting
# =============================================================================

def plot_quotient_pairs(quot):
    """Beta and gamma estimates from each L pair with bootstrap error bars."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Quotient method — estimate per L pair", fontsize=11)

    for ax, key, err_key, mean_key, tv, sym in [
        (axes[0], "beta_pairs",  "beta_errs",  "beta",  THEORY["beta"],  "β"),
        (axes[1], "gamma_pairs", "gamma_errs", "gamma", THEORY["gamma"], "γ"),
    ]:
        vals   = quot[key]
        errs   = quot[err_key]
        labels = quot["pair_labels"][:len(vals)]
        if vals:
            ax.errorbar(range(len(vals)), vals, yerr=errs,
                        fmt="o-", ms=8, color="C0", capsize=5,
                        label="Quotient ± bootstrap")
            ax.axhline(tv, color="red", ls="--", lw=1.5,
                       label=f"Theory = {tv:.4f}")
            if len(vals) > 1:
                mean_val = quot[mean_key]
                ax.axhline(mean_val, color="C1", ls=":", lw=1.2,
                           label=f"Weighted mean = {mean_val:.4f}")
            ax.set_xticks(range(len(labels)))
            ax.set_xticklabels(labels, fontsize=9)
        _style(ax, "L pair", sym,
               f"Quotient method: {sym} per doubling pair")

    fig.tight_layout()
    return fig


def plot_mle_scan(mle_res):
    """Truncated MLE tau vs smin — plateau and cluster size distribution."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Truncated MLE tau — smin stability scan", fontsize=11)

    smin_vals  = mle_res["smin_vals"]
    tau_vals   = mle_res["tau_vals"]
    sigma_vals = mle_res["sigma_vals"]
    valid      = np.isfinite(tau_vals)

    # Left: tau vs smin
    ax = axes[0]
    ax.errorbar(smin_vals[valid], tau_vals[valid],
                yerr=sigma_vals[valid], fmt="o-", ms=5,
                color="C0", label="Truncated MLE τ(s_min)", capsize=3)
    ax.axhline(THEORY["tau"], color="red", ls="--", lw=1.5,
               label=f"Theory τ = {THEORY['tau']:.4f}")
    ax.axhline(mle_res["tau"], color="C1", ls=":", lw=1.5,
               label=f"Best estimate = {mle_res['tau']:.4f}")
    ax.axvline(mle_res["best_smin"], color="green", ls=":", lw=1.2,
               label=f"Best s_min = {mle_res['best_smin']}")
    ax.axhspan(THEORY["tau"] * 0.95, THEORY["tau"] * 1.05,
               alpha=0.08, color="red", label="±5% band")
    ax.set_xscale("log")
    _style(ax, "s_min", "τ (truncated MLE)",
           "Truncated MLE tau vs s_min — stable plateau = best estimate")

    # Right: cluster size distribution with MLE fit overlay
    ax = axes[1]
    sizes = mle_res["sizes"]
    bins  = np.logspace(np.log10(2), np.log10(sizes.max()), 50)
    counts, edges = np.histogram(sizes, bins=bins)
    centres = np.sqrt(edges[:-1] * edges[1:])
    mask    = counts > 0
    ax.loglog(centres[mask], counts[mask], ".", ms=4, color="C0",
              alpha=0.6, label="observed n_s")

    tau_best = mle_res["tau"]
    smin     = mle_res["best_smin"]
    smax     = mle_res["smax"]
    s_fit    = np.logspace(np.log10(smin), np.log10(smax * 0.4), 200)
    # Normalise by matching at a reference point near smin
    ref_idx = np.argmin(np.abs(centres[mask] - smin * 2))
    ref_s   = centres[mask][ref_idx]
    ref_c   = counts[mask][ref_idx]
    norm    = ref_c * (ref_s / s_fit[0]) ** tau_best
    ax.loglog(s_fit, norm * (s_fit[0] / s_fit) ** tau_best,
              "r--", lw=1.5, label=f"Truncated MLE τ={tau_best:.3f}")
    ax.axvline(smax, color="grey", ls=":", lw=1,
               label=f"s_max = L² = {int(smax)}")

    _style(ax, "Cluster size s", "Count",
           f"Cluster size distribution at p≈pc  (L={L_MLE})")

    fig.tight_layout()
    return fig


def plot_summary_comparison(quot, mle_res, pc, nu):
    """
    Summary figure: grouped bar chart (left) + formatted table (right).
    Matches the style of the main run summary figure.
    """
    fig = plt.figure(figsize=(14, 6))
    fig.suptitle(
        f"2-D Site Percolation — Critical Exponent Summary  "
        f"(L_main={L_MLE}, L_FSS={L_FSS})",
        fontsize=11, fontweight="bold",
    )

    # --- Bar chart ---
    ax_bars = fig.add_axes([0.04, 0.12, 0.58, 0.75])

    exponents = [
        ("β", quot["beta"],    quot["beta_std"],   THEORY["beta"]),
        ("γ", quot["gamma"],   quot["gamma_std"],  THEORY["gamma"]),
        ("τ", mle_res["tau"],  mle_res["sigma"],   THEORY["tau"]),
        ("ν", nu,              np.nan,             THEORY["nu"]),
    ]

    x     = np.arange(len(exponents))
    width = 0.32
    ax_bars.bar(x - width/2, [e[1] for e in exponents], width,
                color="C0", alpha=0.75, label="Estimated")
    ax_bars.bar(x + width/2, [e[3] for e in exponents], width,
                color="red", alpha=0.35, label="Theory (2-D)")

    for i, (sym, val, err, tv) in enumerate(exponents):
        if np.isfinite(val):
            if np.isfinite(err):
                ax_bars.errorbar(i - width/2, val, yerr=err,
                                 fmt="none", color="black", capsize=4, lw=1.5)
            ax_bars.text(i - width/2, val + tv * 0.04,
                         f"{val:.4f}", ha="center", fontsize=8,
                         fontweight="bold", color="navy")
        ax_bars.text(i + width/2, tv + tv * 0.04,
                     f"{tv:.4f}", ha="center", fontsize=8, color="darkred")

    ax_bars.set_xticks(x)
    ax_bars.set_xticklabels([e[0] for e in exponents], fontsize=12)
    ax_bars.set_ylabel("Value")
    ax_bars.set_title("Estimated vs theoretical exponents")
    ax_bars.legend(fontsize=9)
    ax_bars.grid(True, axis="y", alpha=0.3)

    # --- Table ---
    ax_table = fig.add_axes([0.66, 0.08, 0.32, 0.82])
    ax_table.axis("off")

    rows = [
        ("beta",  quot["beta"],   quot["beta_std"],   THEORY["beta"],  "Quotient"),
        ("gamma", quot["gamma"],  quot["gamma_std"],  THEORY["gamma"], "Quotient"),
        ("tau",   mle_res["tau"], mle_res["sigma"],   THEORY["tau"],   "Trunc. MLE"),
        ("nu",    nu,             np.nan,             THEORY["nu"],    "FSS / theory"),
        ("pc",    pc,             np.nan,             THEORY["pc"],    "FSS"),
    ]

    col_labels = ["Exp", "Estimated", "Theory"]
    cell_text  = []
    for exp, val, err, tv, method in rows:
        err_str = f" ±{err:.4f}" if np.isfinite(err) else ""
        est_str = f"{val:.4f}{err_str}" if np.isfinite(val) else "—"
        cell_text.append([exp, est_str, f"{tv:.4f}"])

    table = ax_table.table(
        cellText  = cell_text,
        colLabels = col_labels,
        cellLoc   = "center",
        loc       = "center",
        bbox      = [0, 0.05, 1, 0.90],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    for j in range(len(col_labels)):
        table[0, j].set_facecolor("#2c5f8a")
        table[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        colour = "#ddeeff" if i % 2 == 0 else "white"
        for j in range(len(col_labels)):
            table[i, j].set_facecolor(colour)

    ax_table.set_title("Summary table", fontsize=9, pad=4)
    return fig


# =============================================================================
# Terminal summary
# =============================================================================

def print_summary(quot, mle_res, pc, nu):
    print("\n" + "="*72)
    print("  Final exponent summary")
    print("="*72)
    print(f"  {'Exp':<8} {'Estimated':>12} {'Uncertainty':>13} "
          f"{'Theory':>10} {'% error':>9}  Method")
    print("-"*72)

    rows = [
        ("beta",  quot["beta"],   quot["beta_std"],   THEORY["beta"],  "Quotient"),
        ("gamma", quot["gamma"],  quot["gamma_std"],  THEORY["gamma"], "Quotient"),
        ("tau",   mle_res["tau"], mle_res["sigma"],   THEORY["tau"],   "Trunc. MLE"),
        ("nu",    nu,             np.nan,             THEORY["nu"],    "FSS / theory"),
        ("pc",    pc,             np.nan,             THEORY["pc"],    "FSS"),
    ]

    for exp, val, err, tv, method in rows:
        pct     = abs(val - tv) / tv * 100 if np.isfinite(val) else np.nan
        err_str = f"± {err:.4f}" if np.isfinite(err) else "  —      "
        pct_str = f"{pct:.1f}%" if np.isfinite(pct) else "  —  "
        print(f"  {exp:<8} {val:>12.5f} {err_str:>13} "
              f"{tv:>10.4f} {pct_str:>9}  {method}")

    print("="*72)
    print(f"\n  tau MLE details: smin={mle_res['best_smin']}  "
          f"n={mle_res['best_n']:,} clusters  "
          f"smax=L²={int(mle_res['smax'])}")
    nu_note = "exact theory" if abs(nu - 4/3) < 1e-6 else "measured FSS"
    print(f"  nu used: {nu:.6f} ({nu_note})")


# =============================================================================
# Main
# =============================================================================

def main():
    print("=" * 60)
    print("  Exponent analysis — quotient method + truncated MLE tau")
    print("=" * 60)
    nu_note = "exact theory" if abs(NU_OVERRIDE - 4/3) < 1e-6 else "measured FSS"
    print(f"  pc = {PC_OVERRIDE:.5f}   nu = {NU_OVERRIDE:.6f}  ({nu_note})")
    print(f"  L_FSS = {L_FSS}   L_MLE = {L_MLE}")
    print(f"  RUNS_QUOTIENT = {RUNS_QUOTIENT}   RUNS_MLE = {RUNS_MLE}")
    print(f"  N_BOOT = {N_BOOT}   SMIN_FLOOR = {SMIN_FLOOR}")

    pc = PC_OVERRIDE
    nu = NU_OVERRIDE

    # ------------------------------------------------------------------
    # Quotient method — beta and gamma
    # ------------------------------------------------------------------
    print(f"\n[1/3] Quotient method (beta + gamma) ...")
    quot = run_quotient(pc, nu)
    print(f"\n  beta  = {quot['beta']:.4f} ± {quot['beta_std']:.4f}"
          f"  (theory {THEORY['beta']:.4f})")
    print(f"  gamma = {quot['gamma']:.4f} ± {quot['gamma_std']:.4f}"
          f"  (theory {THEORY['gamma']:.4f})")

    # ------------------------------------------------------------------
    # Truncated MLE — tau
    # ------------------------------------------------------------------
    print(f"\n[2/3] Truncated MLE tau ...")
    mle_res = run_mle_tau(pc)

    # ------------------------------------------------------------------
    # Summary + figures
    # ------------------------------------------------------------------
    print(f"\n[3/3] Saving figures ...")
    print_summary(quot, mle_res, pc, nu)

    _save(plot_quotient_pairs(quot),                        "quotient_pairs")
    _save(plot_mle_scan(mle_res),                          "mle_tau_scan")
    _save(plot_summary_comparison(quot, mle_res, pc, nu),  "exponent_comparison")

    print(f"\nDone. Figures saved to: {FIG_DIR}/")
    print("Keep all other figures from your overnight run unchanged.")


if __name__ == "__main__":
    main()

  Exponent analysis — quotient method + truncated MLE tau
  pc = 0.59296   nu = 1.378000  (measured FSS)
  L_FSS = [16, 32, 64, 128, 256, 512]   L_MLE = 512
  RUNS_QUOTIENT = 1000   RUNS_MLE = 2000
  N_BOOT = 300   SMIN_FLOOR = 5

[1/3] Quotient method (beta + gamma) ...

  P window: [0.58996, 0.59596]  7 points
  Using p = 0.59296  (offset from pc = +0.000000)


  Quotient L=  16: 100%|██████████| 1000/1000 [00:03<00:00]


    L=  16  Pinf=0.22195  chi=28.59


  Quotient L=  32: 100%|██████████| 1000/1000 [00:13<00:00]


    L=  32  Pinf=0.20775  chi=89.68


...

    L= 256  Pinf=0.15092  chi=3458.28


  Quotient L= 512: 100%|██████████| 1000/1000 [49:48<00:00]


    L= 512  Pinf=0.15289  chi=11244.03
    beta  L=16->32: 0.1314 ± 0.0910  (theory 0.1389)
    gamma L=16->32: 2.2729 ± 0.0704  (theory 2.3889)
    beta  L=32->64: 0.1199 ± 0.0881  (theory 0.1389)
    gamma L=32->64: 2.3212 ± 0.0784  (theory 2.3889)
    beta  L=64->128: 0.3487 ± 0.0930  (theory 0.1389)
    gamma L=64->128: 2.4607 ± 0.0776  (theory 2.3889)
    beta  L=128->256: 0.1667 ± 0.0991  (theory 0.1389)
    gamma L=128->256: 2.4788 ± 0.0847  (theory 2.3889)
    beta  L=256->512: -0.0258 ± 0.0852  (theory 0.1389)
    gamma L=256->512: 2.3440 ± 0.0793  (theory 2.3889)

  beta  = 0.1406 ± 0.0407  (theory 0.1389)
  gamma = 2.3683 ± 0.0347  (theory 2.3889)

[2/3] Truncated MLE tau ...

  L=512, runs=2000, p=0.59296, smax=L²=262144, smin_floor=5


  MLE tau  L=512: 100%|██████████| 2000/2000 [1:41:06<00:00]


  Total non-spanning clusters (s≥2): 6,158,576
  tau = 1.8988 ± 0.0005  smin=5  n=2,806,964 clusters  (theory 2.0549)

[3/3] Saving figures ...

  Final exponent summary
  Exp         Estimated   Uncertainty     Theory   % error  Method
------------------------------------------------------------------------
  beta          0.14061      ± 0.0407     0.1389      1.2%  Quotient
  gamma         2.36828      ± 0.0347     2.3889      0.9%  Quotient
  tau           1.89881      ± 0.0005     2.0549      7.6%  Trunc. MLE
  nu            1.37800       —           1.3333      3.3%  FSS / theory
  pc            0.59296       —           0.5927      0.0%  FSS

  tau MLE details: smin=5  n=2,806,964 clusters  smax=L²=262144
  nu used: 1.378000 (measured FSS)
  Saved: figures/quotient_pairs.png
  Saved: figures/mle_tau_scan.png
  Saved: figures/exponent_comparison.png

Done. Figures saved to: figures/
Keep all other figures from your overnight run unchanged.
